<a href="https://colab.research.google.com/github/GabrielEganStudent/hello-world/blob/main/Distant_reading_from_the_command_line.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distant reading from the command line

## Getting a corpus via curl

In [ ]:
#@title Download corpus

# url for corpus
url_txt = 'https://www.dropbox.com/scl/fi/q0l112qwmyvw0g3rbw4a0/dialnarr_txt.zip?rlkey=phl6tlza2d9awhm7o7kjn9ma8&st=woyfev5r&dl=1'

# curl to download
# -L = follow redirects (needed for dropbox links)
# -C - = resume/do not overwrite download
# -o = save as filename
!curl -L -C - -o corpus_txt.zip "$url_txt"

In [ ]:
#@title Unzip corpus zip file

# unzip
# -n = do not overwrite if exist already
# -q = quiet

!unzip -q -n corpus_txt.zip

In [ ]:
#@title Challenge: Open the terminal (bottom left button) and ls the txt/ folders


## Exploring corpus via grep

In [ ]:
#@title Grep for keyword: count results in dialogue vs narration

# set keyword (param gives right-hand side text box in Colab)
keyword = 'socialism' #@param {type: "string"}

# print matches in narration
!echo "Matches in narration:"

# grep
# -i = case insensitive
# -r = follow directory recursively
# wc -l = count lines in piped grep output
!grep -i -r $keyword txt/narration | wc -l

In [ ]:
#@title Challenge: do same for dialogue
!echo "Matches in dialogue:"


In [ ]:
#@title Show matches for narration

# grep
# -C 2 = show 2 lines before and after match
!grep -i -r -C 2 "$keyword" txt/narration | head -n50

# do the same with sorted filenames
# !find txt/narration -name "*.txt" -type f | sort | xargs grep -i -C 2 "$keyword" | head -n50

In [ ]:
#@title Show matches for dialogue

!grep -i -r -C 2 "$keyword" txt/dialogue | head -n50

# Do the same with sorted filenames
# !find txt/dialogue -name "*.txt" -type f | sort | xargs grep -i -C 2 "$keyword" | head -n50

## To Python!

In [ ]:
#@title Tokenizer: text string -> words

# regular expressions in python
import re

# a simple fast tokenizer that treats punctuation as a separate token
def tokenize_fast(txt):
    # [\w']+ = matches one or more word characters (letters, digits, underscore) plus apostrophes
    # [.,!?; -—–\n] = matches individual punctuation
    # txt.lower() = lowercases the input
    return re.findall(r"[\w']+|[.,!?;-—–]", txt.lower())

# demo
tokenize_fast("It was the best of times, it was the worst of times. But here's—some, punctuation!")

In [ ]:
#@title Keywords in context
import nltk,os,subprocess

def kwic(keyword, folder='txt/', limit=3, min_count=2):
    # use grep to find matching filenames (since it is much faster than Python)

    # -l = return only filenames
    !grep -i -l -r $keyword $folder > matches.txt

    # Open list of filenames to loop over
    with open('matches.txt') as f:
        matching_filenames = f.read().splitlines()

    # For each filename in the list (up to limit)
    done = 0
    for fn in sorted(matching_filenames):
        # Open the file
        with open(fn) as f:
            # Read the file to a string called `txt`
            txt=f.read()

        # Check if our keyword is in the text (ignore case) more than min_count
        if txt.lower().count(keyword.lower()) >= min_count:
            # Print the filename of this text
            print(fn)

            # Tokenize the text into a list of words
            words = tokenize_fast(txt)

            # Pass this list of words to nltk's Text object
            nltk_text = nltk.Text(words)

            # Use NLTK to produce a concordance
            nltk_text.concordance(keyword, width=140)

            # Print a final blank line
            print('\n')

            # break the loop if past the limit
            done += 1
            if done >= limit:
                break

In [ ]:
#@title Run KWIC on narration for our keyword
kwic(keyword, 'txt/narration')

In [ ]:
#@title Run KWIC on dialogue for our keyword
kwic(keyword, 'txt/dialogue')

## Plotting counts

In [ ]:
#@title Get list of all matching filenames

# -c = return counts
# grep -v ":0$" = exclude lines with 0 count (i.e. lines ending :0)
!grep -i -c -r "$keyword" txt/ | grep -v ":0$" > matches_with_counts.txt

# show the first 10 lines of this file
!head -n10 matches_with_counts.txt

In [ ]:
#@title Parse list of filenames into data
import pandas as pd
from tqdm.auto import tqdm


def get_count_data(keyword, folder='txt'):
    # (same as above) find counts for keyword in folder
    !grep -i -c -w -r "$keyword" "$folder" | grep -v ":0$" > matches_with_counts.txt

    # read the file grep made
    with open('matches_with_counts.txt') as f:
        txt = f.read()

    # start an empty list for our results
    all_results = []

    # loop over lines in the grep-made file
    for line in tqdm(txt.splitlines()):

        # get the filepath and count part of line
        filepath,count = line.split(':')

        # get the filename from the filepath (after the last /)
        filename = line.split('/')[-1]

        # [corpus-specific patch for some files]
        filename = filename.replace('.ascii.txt','')

        # these filenames should follow a standardized format for info separated by periods
        try:
            year,author,nationality,gender,title,narr_or_dial,ext = filename.split('.')
        except Exception as e:
            # just in case...
            print(f'!! Error parsing filename: {filename}')
            continue

        # get the total count for this file
        with open(filepath) as f:
            total_words = len(f.read().split())

        # make a dictionary to store our results
        result = {
            'year': int(year),
            'author': author,
            'nationality': nationality,
            'gender': gender,
            'title': title.replace('_', ' '),
            'register': narr_or_dial,
            'count': int(count),
            'num_words': total_words,
            'filename': filename,
        }

        # append this result dictionary to results list
        all_results.append(result)

    # turn the list of all results into a pandas DataFrame (pandas = excel for python)
    df = pd.DataFrame(all_results)

    # return dataframe
    return df

# run our function on our keyword
df = get_count_data(keyword)

# show the dataframe
df

In [ ]:
#@title Sort the dataframe
df.sort_values('count',ascending=False)

In [ ]:
#@title Summing by category
df_sums = df.groupby(['register'])['count'].sum()
df_sums

In [ ]:
#@title Summing by category while normalizing
df_sums2 = df.groupby(['register'])[['count','num_words']].sum()
df_sums2['fpm'] = df_sums2['count']/df_sums2['num_words']*1_000_000
df_sums2

In [ ]:
#@title Challenge: sum by category for nationality



In [ ]:
#@title Challenge: sum by category for nationality AND register


## Plotting data with python

In [ ]:
#@title Plotting individual texts across years

# we'll use plotly express for plotting (pre-installed on Colab)
import plotly.express as px

# our plotting function
def plot_by_year(df):
    # make a scatterplot
    fig = px.scatter(
        # our data
        df,

        # x-axis
        x='year',

        # y-axis
        y='count',

        # color by
        color='register',
        color_discrete_map={'dialogue': 'red', 'narration': 'blue'},

        # facet by
        facet_col='nationality',

        # show metadata on hover
        hover_data=df.columns,

        # title of plot
        title=f'Count of "{keyword}" by year and nationality'
    )

    # show the figure
    fig.show()

# run our plotting function
plot_by_year(df)

In [ ]:
#@title Plotting by decade/period sums

def plot_by_period(df, period_length=10):
    # copy our dataframe before we modify it
    df = df.copy()

    # Set the period for each row
    df['period'] = df['year']//period_length*period_length

    # Get our counts by period + register + nationality
    df_counts = df.groupby(['period', 'register','nationality'])['count'].sum().reset_index()

    # Make our plot
    fig = px.line(
        df_counts,
        x='period',  # this time period is the x-axis
        y='count',
        color='register',
        facet_col='nationality',
        color_discrete_map={'dialogue': 'red', 'narration': 'blue'},
        title=f'Count of "{keyword}" by {period_length}-year periods and nationality'
    )
    fig.show()

# run the plot function
plot_by_period(df)

In [ ]:
#@title Challenge: Rerun for another keyword
keyword = 'london' # @param {type:"string"}

# run get_count_data on our keyword
# df =

# run plot_by_period on df

# run plot_by_year on df

# run kwic on keyword, txt/narration, limit=1

# run kwic on keyword, txt/dialogue, limit=1

## Most distinctive words

In [ ]:
#@title Counting all words in texts

# a special dictionary to store counts (key values default to 0)
from collections import Counter

# a function to get a Counter for all words in a text file
def get_word_counts_in_file(filepath):
    # start an empty counts
    counts = Counter()

    # open file
    with open(filepath) as f:
        # read the text
        txt = f.read()

    # loop over each word
    for word in tokenize_fast(txt):
        # add 1 to the count for that word
        counts[word]+=1

    return counts


# a function to get a big Counter for word counts from all texts in a folder
def get_word_counts(folder):
    # start an empty counts
    counts = Counter()

    # loop over filenames in the folder
    for fn in tqdm(list(os.listdir(folder)), desc=f'Counting words in {folder}'):

        # get counts in file
        counts_in_file = get_word_counts_in_file(os.path.join(folder, fn))

        # update counts for folder based on counts in file
        counts.update(counts_in_file)

    # return the counts as Counter
    return counts


In [ ]:
#@title Running word counts

# collect word counts for all dialogue texts
dialogue_words = get_word_counts('txt/dialogue')

# collect word counts for all narration texts
narration_words = get_word_counts('txt/narration')

In [ ]:
#@title Finding most distinctive words

# using fisher exact test from scipy stats (good for non-normalized count data)
from scipy.stats import fisher_exact

# defining a "mdw" function accepting two Counters and a parameter for top n words to consider
def mdw(counts1, counts2, topn=5000):

    # get total word count in both counters
    total_words1 = sum(counts1.values())
    total_words2 = sum(counts2.values())

    # get a dictionary of the counts for the top n words
    top_word_counts1 = dict(counts1.most_common(topn))
    top_word_counts2 = dict(counts2.most_common(topn))

    # find the words present in both top n of counters (set intersection)
    common_words = set(top_word_counts1.keys()) & set(top_word_counts2.keys())

    # make an empty list to store results
    results = []

    # loop over each word in the set of common words
    for word in tqdm(common_words):
        # get count for this word
        count1 = top_word_counts1[word]
        count2 = top_word_counts2[word]

        # make a contingency table of count1 | count2 || total1 | total2  (i.e. present1 | present2 || absent1 | absent2)
        contingency_table = [
            [
                count1,
                count2,
            ],
            [
                total_words1-count1,   # total - count = absent
                total_words2-count2
            ]
        ]

        # run fisher exact test, get odds ratio and p-value
        oddsratio, pvalue = fisher_exact(contingency_table)

        # make a result dictionary
        result = {
            'word': word,
            'oddsratio': oddsratio,
            'pvalue': pvalue,
            'count1': count1,
            'count2': count2,
            'fpm1': count1/total_words1 * 1_000_000,
            'fpm2': count2/total_words2 * 1_000_000,
            'total1': total_words1,
            'total2': total_words2,
        }

        # add result to list of results
        results.append(result)

    # return as pandas dataframe
    return pd.DataFrame(results).sort_values('oddsratio')

In [ ]:
#@title Running MDW
df_mdw = mdw(dialogue_words, narration_words)
df_mdw

In [ ]:
#@title Show top 25 most narration words
df_mdw.head(25)

In [ ]:
#@title Challenge: Show top 25 most dialogue words


## Find most American vs. British words in dialogue

In [ ]:

# Another function to get word counts but with a filename filter applied (.Br. = British, .Am. = American)
def get_word_counts_specific(folder, filename_filter='.Br.'):
    # start counter
    counts = Counter()
    # for each filename in folder
    for fn in tqdm(list(os.listdir(folder)), desc=f'Counting words in {folder}'):
        # make sure filename filter in filename
        if not filename_filter in fn:
            continue
        # Read text
        with open(os.path.join(folder, fn)) as f:
            txt = f.read()
        # Add words to counter
        for word in tokenize_fast(txt):
            counts[word]+=1
    # return the counter
    return counts

In [ ]:
british_dialogue_words = get_word_counts_specific('txt/dialogue', '.Br.')
american_dialogue_words = get_word_counts_specific('txt/dialogue', '.Am.')

In [ ]:
df_mdw_br_am_dialogue = mdw(american_dialogue_words, british_dialogue_words)
df_mdw_br_am_dialogue

In [ ]:
#@title Challenge: Show top 25 most British dialgue words



In [ ]:
#@title Challenge: Show top 25 most American dialgue words



In [ ]:
#@title Challenge: Find most American vs. British words in narration



In [ ]:
#@title Challenge: Find most American vs. British words in narration



In [ ]:
#@title Challenge: Plot a word of interest by year and decade



In [ ]:
#@title Challenge: Show KWIC for word of interest in dialogue and in narration



In [ ]:
#@title Super challenge: What are the distinctive words in American and British narration *in the 2 lines surrounding mentions of London*?

# Use grep + python
